# Chains in Langchain: 
hains are fundamental, modular sequences of components (like LLMs, prompts, tools, parsers) linked together to automate multi-step AI workflows, passing output from one step as input to the next, much like a "train of thought" to build complex applications from simple tasks, with common types including LLMChain (basic prompt + model) and SequentialChain (linking multiple sub-chains). 

In [1]:
# model calling through Huggingfacehiub
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
load_dotenv()
model =ChatHuggingFace(llm=HuggingFaceEndpoint(
        repo_id="openai/gpt-oss-20b",
        task="text-generation",
        huggingfacehub_api_token=os.getenv("HUGGINFACE_API_KEY"))
    )
model1 =ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id='google/gemma-2-2b-it',
    task='text-generation',
    huggingfacehub_api_token=os.getenv("HUGGINFACE_API_KEY")
))

/Users/vinod/DaasAI/GenAI_pract/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Squenntial chains: 
chain in sequence A--B--C

In [2]:
#Simple Chain 
prompt1 = PromptTemplate(
    template=""" Describe about the following topic in detail:\n
    {topic}
    """, 
    input_variables=["topic"]
)
parser =StrOutputParser()

chain =prompt1 | model | parser
response =chain.invoke({"topic": "Artificial Intelligence"})
print(response)

## Artificial Intelligence (AI) – A Comprehensive Overview

### 1. What Is Artificial Intelligence?

Artificial Intelligence is a multidisciplinary field of computer science that seeks to create systems capable of performing tasks that, today, require human intelligence. These tasks include **reasoning, learning, perception, language understanding, decision‑making, and problem solving**. AI systems can range from simple rule‑based scripts to complex networks that learn from massive data streams.

| Term | Rough Meaning | Example |
|------|---------------|---------|
| **Narrow/Weak AI** | Systems that excel at a single task | Voice assistants (Siri, Alexa) |
| **General/Strong AI** | Systems with human‑like intelligence across domains | Hypothetical AGI (Artificial General Intelligence) |
| **Super‑AI** | Intelligence surpassing human cognition | Speculative future state |

### 2. Historical Milestones

| Year | Milestone | Significance |
|------|-----------|--------------|
| **1950** |

In [3]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


In [4]:
# sequential Chain

prompt2 = PromptTemplate(
    template=""" Please generate question and answers from given text:\n
    {text}
    """,
    input_variables=["text"]
)
chain2 =prompt1 | model | parser | prompt2 | model | parser

chain2_response =chain2.invoke({"topic": "Machine Learning"})
print(chain2_response)


**Sample Question & Answer Set (based on the provided text)**  

---

### 1. Foundations  

| # | Question | Answer |
|---|----------|--------|
| 1 | **What is the main difference between AI and ML?** | AI is a broad discipline aiming to create systems that perform tasks requiring human intelligence. ML is a subset of AI that focuses on algorithms that automatically improve by learning from data rather than being explicitly programmed. |
| 2 | **Why is data crucial for machine learning?** | The quality and quantity of data directly influence a model’s performance. Good data enables the algorithm to learn accurate patterns, while poor data can lead to overfitting, bias, or underperformance. |
| 3 | **What constitutes a “model” in machine learning?** | A model is the internal representation that an algorithm learns from data—often a set of weights or parameters. Models can range from simple linear regressions to complex deep neural networks. |
| 4 | **Give an example of a simple machine‑

# Parallel Chain :

```
A--------------------B
   parallel chain        >>merge the chain 
C--------------------D
```

In [12]:
# we will create a chain that will create and summery of topic and create job oporutnies and bussiness oportunity for these topics
# we will creater pydantics parser for the final output
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional
class Report(BaseModel):
    report: str = Field(..., description="Detailed report based on the topic summary and business opportunities.")
    best_5_business_opportunities: Optional[str] = Field(..., description="List of the best 5 business opportunities.")

pydanticsparser = PydanticOutputParser(pydantic_object=Report)

prompt3 = PromptTemplate(
    template=""" Generate a detailed summary for the following topic:\n
    {topic}
    """,
    input_variables=["topic"]
)   

prompt4 = PromptTemplate(
    template="""Generate potential job opportunities and business ideas on this topic:\n
    {topic}
    """,
    input_variables=["topic"]
)

prompt5 = PromptTemplate(
    template=""" Merge summary: {summary} and bussiness ooportunities: {bussiness_opportunities} and create a detailed report and best_5_business_opportunities.
    \n 
    """,
    input_variables=["summary", "bussiness_opportunities"],
    
    # partial_variables={
    #     "format_instructions": pydanticsparser.get_format_instructions()
    #     }
)

parallel_chain = RunnableParallel(
    {'summary': prompt3 | model | parser,
    'bussiness_opportunities': prompt4 | model1 | parser
})
merge_chain =prompt5 | model | parser


final_par_chain =parallel_chain | merge_chain
final_response =final_par_chain.invoke({"topic": "Artificial Intelligence"})
print(final_response)



# Artificial Intelligence – Detailed Summary & Business Outlook

---

## Executive Summary  

Artificial Intelligence (AI) has evolved from a speculative academic pursuit in the 1950s to a ubiquitous, high‑impact technology that permeates every major industry today. Its core promise—machines that can learn, reason, and act like humans—has been realized through a blend of rule‑based systems, statistical learning, deep neural networks, and large‑scale data pipelines.  

The AI ecosystem now supports a vibrant job market and a rich array of business opportunities. Companies that can harness AI’s power for **data‑driven decision making, automation, personalization, and predictive analytics** are positioned to capture significant value. Below, we merge the foundational overview with actionable insights for professionals and entrepreneurs alike.

---

## 1. What is Artificial Intelligence?

- **Definition**: A scientific and engineering discipline focused on creating non‑biological systems c

In [ ]:
import json 
with open("final_response.json", "w") as f:
    json.dump(final_response.model_dump(), f, indent=4)

In [ ]:
print(final_response.report)

Artificial Intelligence (AI) has evolved from a theoretical concept in the 1950s to a ubiquitous technology that powers a wide range of modern applications. The field began with foundational ideas such as Alan Turing’s question about machine thought and was formally introduced at the 1956 Dartmouth Conference. Over subsequent decades, AI progressed through rule‑based expert systems, the advent of machine learning and back‑propagation, and landmark achievements like IBM’s Deep Blue and Google’s AlphaGo. The 2010s brought a deep‑learning renaissance with AlexNet, and the 2020s are dominated by large foundation models such as GPT‑4, diffusion models for image generation, and multimodal systems that combine vision, language, and audio.

Core AI paradigms now include symbolic AI, which uses logic and rule‑based reasoning; statistical/connectionist AI, which relies on neural networks; reinforcement learning, which trains agents via reward signals; and hybrid neuro‑symbolic approaches that me

In [ ]:
print(final_response.best_5_business_opportunities)

['AI‑powered Content Creation']


In [11]:
final_par_chain.get_graph().print_ascii()

   +------------------------------------------------+    
   | Parallel<summary,bussiness_opportunities>Input |    
   +------------------------------------------------+    
                  ***               ***                  
               ***                     ***               
             **                           **             
+----------------+                    +----------------+ 
| PromptTemplate |                    | PromptTemplate | 
+----------------+                    +----------------+ 
          *                                   *          
          *                                   *          
          *                                   *          
+-----------------+                  +-----------------+ 
| ChatHuggingFace |                  | ChatHuggingFace | 
+-----------------+                  +-----------------+ 
          *                                   *          
          *                                   *          
          *   

# Conditional Chains :
: According to contion like if -else case this will be going to be executed here.


In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
 

parser = StrOutputParser()
 
class feedback(BaseModel):
    feedback: Literal["positive", "negative"] = Field(..., description="User feedback on the report.")
pydanticsparser = PydanticOutputParser(pydantic_object=feedback)

prompt6 = PromptTemplate(
    template=""" Based on the report: {report}, classify the feedback as positive or negative.
    \n {format_instructions}
    """,
    input_variables=["report"],
    partial_variables={
        "format_instructions": pydanticsparser.get_format_instructions()
        }
)   

prompt7=PromptTemplate(
    template="""write appropriate response on this report beased on user positive feedback on report {report}""",
    input_variables=["report"]  
)
prompt8=PromptTemplate(
    template ="""write appropriate response on this report beased on user negative feedback on report {report}""",
    input_variables=["report"]
)

chain_feedback = prompt6 | model | pydanticsparser

branch_chain = RunnableBranch(
    (lambda x: x.feedback == "positive", prompt7 | model | parser),
    (lambda x: x.feedback == "negative", prompt8 | model | parser),
    RunnableLambda(lambda x: "No valid feedback provided.")
)
chain_final_feedback = chain_feedback | branch_chain
input_feedback=input("Please provide your feedback on the report (positive/negative): ")
final_feedback_response = chain_final_feedback.invoke({"report": input_feedback})
print(final_feedback_response)

chain.get_graph().print_ascii() 


Thank you for taking the time to review the report and share your positive feedback. I'm glad to hear that the insights and recommendations resonated with you. Your support encourages us to continue delivering high‑quality analysis and actionable guidance. If there are any additional areas you'd like us to explore further or if you have specific next steps in mind, please let me know. Looking forward to our continued collaboration!
     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
   +-----------------+     
   | ChatHuggingFace |     
   +-----------------+     
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +